## DATA CAPTURE FROM CAMERA

• CREATE CAMERA CLASS.   
• INITIALISE SETUP.    
• CREATE CAPTURE FUNCTION.    
• SAVE VIDEO CAPTURED.   

In [ ]:
'''--- IGNORE ---
import os
from pathlib import Path

# --- FIND GIT REPOSITORY ROOT ---
def find_repo_root():
    path = Path.cwd().resolve()
    
    for parent in [path] + list(path.parents):
        if (parent / ".git").exists():
            return parent
    
    raise RuntimeError("Git repository root not found")


REPO_ROOT = find_repo_root()
print(f"Repository root found at: {REPO_ROOT}")

# Optional: list files in repo root
print(os.listdir(REPO_ROOT))
'''


In [ ]:
'''--- IGNORE ---
import sys
import os

import sys
sys.path.append(str(REPO_ROOT))
'''

In [ ]:
from robot_utils import (
    crop_frame,
    detect_yellow,
    fit_line,
    get_line_direction,
    get_steering_magnitude,
    get_rope_side
)

In [ ]:
import ipywidgets.widgets as widgets
import motors

robot = motors.MotorsYukon(mecanum=False)
print("Robot is ready:)")

## CAMERA CLASS 

initialise camera    

Core functions:  

• capture frames: get the .avi and .bin files this way      
• start thread       
• stop thread    


In [ ]:
import traitlets
import cv2
import numpy as np
import pyzed.sl as sl
import math
import numpy as np
import sys
import math
import threading
from traitlets.config.configurable import SingletonConfigurable
import time

import ipywidgets.widgets as widgets
from IPython.display import display

#create two widgets for the displaying of the image
display_color = widgets.Image(format='jpeg', width='45%') #determine the width of the color image
display_depth = widgets.Image(format='jpeg', width='45%')  #determine the width of the depth image
layout=widgets.Layout(width='100%')

sidebyside = widgets.HBox([display_color, display_depth],layout=layout) #horizontal display


# display the widget
display(sidebyside) 

timestamp = time.strftime('%Y%m%d_%H%M%S')
# Define a Camera class that inherits from SingletonConfigurable
class Camera(SingletonConfigurable):
    color_value = traitlets.Any() # monitor the color_value variable
    def __init__(self):
        super(Camera, self).__init__()

        self.zed = sl.Camera()
        # Create a InitParameters object and set configuration parameters
        init_params = sl.InitParameters()
        init_params.camera_resolution = sl.RESOLUTION.VGA #VGA(672*376), HD720(1280*720), HD1080 (1920*1080) or ...
        init_params.depth_mode = sl.DEPTH_MODE.NONE  # depth not needed

        # Open the camera
        status = self.zed.open(init_params)
        if status != sl.ERROR_CODE.SUCCESS: #Ensure the camera has opened succesfully
            print("Camera Open : "+repr(status)+". Exit program.")
            self.zed.close()
            exit(1)

         # Create and set RuntimeParameters after opening the camera
        self.runtime = sl.RuntimeParameters()

        #flag to control the thread
        self.thread_runnning_flag = False

        # Get the height and width
        camera_info = self.zed.get_camera_information()
        self.width = camera_info.camera_configuration.resolution.width
        self.height = camera_info.camera_configuration.resolution.height
        self.image = sl.Mat(self.width,self.height,sl.MAT_TYPE.U8_C4, sl.MEM.CPU)

        #setup output file
        fourcc = cv2.VideoWriter_fourcc(*'XVID')
        self.color_writer = cv2.VideoWriter(f'color_video{timestamp}.avi', fourcc, 30, (672, 376))

    def _capture_frames(self): #For data capturing only

        while(self.thread_runnning_flag==True): #continue until the thread_runnning_flag is set to be False
            if self.zed.grab(self.runtime) == sl.ERROR_CODE.SUCCESS:
                
                # Retrieve Left image
                self.zed.retrieve_image(self.image, sl.VIEW.LEFT)

                self.color_value = self.image.get_data()
                self.color_value = cv2.cvtColor(self.color_value, cv2.COLOR_BGRA2BGR)

                 # Save to file
                try:
                    self.color_writer.write(self.color_value)
                except:
                    print("Error writing file")
                    
    def start(self): #start the data capture thread
        if self.thread_runnning_flag == False: #only process if no thread is running yet
            self.thread_runnning_flag=True #flag to control the operation of the _capture_frames function
            self.thread = threading.Thread(target=self._capture_frames) #link thread with the function
            self.thread.start() #start the thread

    def stop(self): #stop the data capture thread
        if self.thread_runnning_flag == True:
            self.color_writer.release()
            self.thread_runnning_flag = False #exit the while loop in the _capture_frames
            self.thread.join() #wait the exiting of the thread       

def bgr8_to_jpeg(value):#convert numpy array to jpeg coded data for displaying 
    return bytes(cv2.imencode('.jpg',value)[1])
    
#create a camera object
camera = Camera()
camera.start() # start capturing the data

#Convert a NumPy array to JPEG-encoded data for display
def bgr8_to_jpeg(value):
    return bytes(cv2.imencode('.jpg',value)[1])


In [ ]:
# Link camera feed to display widget
camera.observe(lambda change: update_display(change['new']), names=['color_value'])

def update_display(frame):
    if frame is not None:
        display_color.value = bgr8_to_jpeg(frame)

read the captured data and process it to get .avi and .bin into frames

In [ ]:
#-- IGNORE --
'''
colour_dir = REPO_ROOT / 'colour_files'

colour_files = sorted([f for f in os.listdir(colour_dir) if f.endswith('.avi')])

for colour_file in colour_files:
    colour_path = colour_dir / colour_file
    colour_frames = utils.read_image_files(str(colour_path))
'''

In [ ]:
#-- IGNORE --
'''
def create_flipped_avi(input_path, output_path):
    """
    Creates a horizontally flipped copy of an AVI file.
    Left turns become right turns and vice versa.
    """
    cap = cv2.VideoCapture(str(input_path))
    if not cap.isOpened():
        print(f"Could not open {input_path}")
        return

    fps    = cap.get(cv2.CAP_PROP_FPS)
    width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    fourcc = cv2.VideoWriter_fourcc(*'XVID')
    out    = cv2.VideoWriter(str(output_path), fourcc, fps, (width, height))

    print(f"Flipping {input_path.name} ({total} frames)...")

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        out.write(cv2.flip(frame, 1))

    cap.release()
    out.release()
    print(f"Saved flipped video to {output_path}")


# Find your clockwise video
clockwise_avi = [f for f in colour_dir.glob("*.avi") 
                 if "clockwise_1" in f.name.lower()][0]

flipped_path = colour_dir / "clockwise_flipped.avi"
create_flipped_avi(clockwise_avi, flipped_path)
'''

In [ ]:

''' --- IGNORE : COMMENTED DEVELOPMENT CODE ---'''

'''
LABEL_NAMES = ["left", "straight", "right"]
raw_dir     = Path("dataset/raw")

# Clear old dataset
if raw_dir.exists():
    shutil.rmtree(raw_dir)

all_records = []

for colour_file in colour_files:
    colour_path   = colour_dir / colour_file
    colour_frames = utils.read_image_files(str(colour_path))
    print(f"Processing {colour_file}: {len(colour_frames)} frames")

    records = extract_from_frames(
        colour_frames,
        str(raw_dir),
        LABEL_NAMES,
        frame_skip=3,
        min_pixels=1000
    )
    all_records.extend(records)
    print(f"  Saved: {len(records)} frames")

# Class distribution
print("\nClass distribution:")
total = 0
for name in LABEL_NAMES:
    count = len(list((raw_dir / name).glob("*.jpg")))
    print(f"  {name}: {count}")
    total += count
print(f"  total: {total}")
'''

In [ ]:
import torch
from torchvision import transforms

LABEL_NAMES = ['left', 'right', 'straight']  # alphabetical — matches ImageFolder

infer_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

def classify_frame(model, frame, device, threshold=0.5):
    """
    frame: BGR numpy array from ZED camera
    returns: label string, confidence float, raw probs
    """
    if frame.shape[2]== 4:
        frame = frame[:, :, :3]
    tensor = infer_transform(frame).unsqueeze(0).to(device)
    with torch.no_grad():
        probs = torch.softmax(model(tensor), dim=1)[0]
    confidence, idx = probs.max(0)
    return LABEL_NAMES[idx.item()], confidence.item(), probs.cpu().numpy()

In [ ]:
from collections import deque
import cv2

def run_robot(model, robot, device,
              base_speed           = 0.45,
              spin_speed           = 0.12,
              conf_threshold       = 0.85,
              micro_threshold      = 0.10,
              micro_frames         = 2,
              max_spin_frames      = 35,
              align_threshold      = 0.20,
              consecutive_required = 9):

    model.eval()

    score_history         = deque(maxlen=micro_frames)
    consecutive_agreement = 0
    align_count           = 0
    in_corner             = False
    corner_frames         = 0
    last_turn             = None
    normal_frames         = 0
    last_rope_side        = None

    # Annotated video writer
    timestamp        = time.strftime('%Y%m%d_%H%M%S')
    fourcc           = cv2.VideoWriter_fourcc(*'XVID')
    annotated_writer = cv2.VideoWriter(
        f'run_{timestamp}.avi', fourcc, 10, (672, 376)
    )

    # Visualization helpers - draw arrows and labels overlay 
    def draw_arrow(img, direction, ox, oy, color, label):
        length = 50
        if direction == 'left':
            end = (ox - length, oy)
        elif direction == 'right':
            end = (ox + length, oy)
        else:
            end = (ox, oy - length)
        if direction in ('left', 'right', 'straight'):
            cv2.arrowedLine(img, (ox, oy), end, color, 2, tipLength=0.3)
        cv2.putText(img, label, (ox - 20, oy + 15),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.4, color, 1)

    def final_to_direction(final):
        if final in ('slight_left', 'left', 'exit_corner'):
            return 'left'
        elif final in ('slight_right', 'right'):
            return 'right'
        else:
            return 'straight'

    print("Starting — hybrid")

    try:
        while True:
            t_start = time.time()

            frame = camera.color_value
            if frame is None:
                continue
            if frame.shape[2] == 4:
                frame = cv2.cvtColor(frame, cv2.COLOR_BGRA2BGR)

            h = frame.shape[0]

            # --- ResNet ---
            label, conf, _ = classify_frame(model, frame, device)

            # --- Geometry ---
            #crop to bottom half
            cropped   = crop_frame(frame)
            #detect yellow pixels in cropped area
            mask      = detect_yellow(cropped)
            #count yellow pixels
            pixels    = int(np.sum(mask > 0))
            #fit line to yellow pixels
            line      = fit_line(mask)
            #calculate steering magnitude from line
            magnitude = get_steering_magnitude(line, cropped.shape)
            #get line direction and score
            direction, score, _ = get_line_direction(line, mask, cropped.shape)
            score_history.append(magnitude)
            #determine side with most yellow pixels
            rope_side = get_rope_side(mask)

            if rope_side is not None:
                last_rope_side = rope_side

            fallback = rope_side or last_rope_side or last_turn or 'right'

            # --- Consecutive agreement ---
            #checks for agreement and increments
            if rope_side is not None and \
               label in ('left', 'right') and \
               rope_side == label and \
               conf >= conf_threshold:
                consecutive_agreement += 1
                last_turn = label
            else:
                consecutive_agreement = 0

            # --- Normal frame counter ---
            if not in_corner:
                normal_frames += 1
            else:
                normal_frames = 0

            # --- Corner trigger ---
            corner_trigger = (
                consecutive_agreement >= consecutive_required and
                not in_corner
            )

            # ============================================================
            # STATE
            # ============================================================
            final     = 'straight'
            alignment = False

            #triggers corner state when enough agreement is reached
            if not in_corner and corner_trigger:
                in_corner             = True
                corner_frames         = 0
                align_count           = 0
                consecutive_agreement = 0
                print(f"CORNER → {last_turn} | conf:{conf:.2f}")

            if in_corner:
                #increments frame to see how long we've been in the corner
                corner_frames += 1
                final = last_turn

                #maintains spin till adequate alignment is achieved
                if abs(magnitude) < align_threshold:
                    align_count += 1
                    if align_count >= 3:
                        in_corner      = False
                        align_count    = 0
                        normal_frames  = 0
                        last_turn      = None
                        final          = 'exit_corner'
                        print("EXIT corner")
                else:
                    align_count = 0

                #if spins for too long, fallback to searching
                if corner_frames > max_spin_frames:
                    in_corner      = False
                    align_count    = 0
                    normal_frames  = 0
                    final          = f'search_{fallback}'
                    print(f"SPIN TIMEOUT | fallback:{fallback}")

                # Motor command
                if final in ('left', 'exit_corner'):
                    robot.spinLeft(speed=spin_speed)
                elif final == 'right':
                    robot.spinRight(speed=spin_speed)
                elif final.startswith('search_'):
                    if fallback == 'left':
                        robot.spinLeft(speed=spin_speed * 0.8)
                    else:
                        robot.spinRight(speed=spin_speed * 0.8)
                else:
                    robot.stop()

            else:
                # Straight following — line magnitude
                recent = list(score_history)
                #microcorrects to align robot to left or right
                if len(recent) == micro_frames and \
                   all(s < -micro_threshold for s in recent):
                    final     = 'slight_left'
                    alignment = True
                    robot.left(speed=base_speed * 0.8)
                elif len(recent) == micro_frames and \
                     all(s > micro_threshold for s in recent):
                    final     = 'slight_right'
                    alignment = True
                    robot.right(speed=base_speed * 0.8)
                else:
                    final = 'straight'
                    robot.forward(speed=base_speed)

            print(f"{'CORNER' if in_corner else 'NORMAL'} | "
                  f"mag:{magnitude:+.3f} | rope:{rope_side} | "
                  f"agree:{consecutive_agreement}/{consecutive_required} | "
                  f"label:{label}({conf:.2f}) | final:{final}")

            # --- Draw overlay ---
            vis = frame.copy()
            vis_bottom = vis[h//2:, :]
            vis_bottom[mask > 0] = [0, 255, 255]
            vis[h//2:, :] = vis_bottom

            if line is not None:
                vx, vy, x0, y0 = line
                hc, wc = cropped.shape[:2]
                if abs(vy) > 1e-6:
                    x_top    = int(x0 + (0 - y0)  * (vx / vy))
                    x_bottom = int(x0 + (hc - y0) * (vx / vy))
                    cv2.line(vis, (x_top, h//2),
                             (x_bottom, h), (0, 165, 255), 2)

            arrow_y = h - 40
            draw_arrow(vis, direction or 'straight',
                       150, arrow_y, (0, 165, 255), 'line')
            draw_arrow(vis, label,
                       336, arrow_y, (255, 200, 0), f'resnet {conf:.2f}')
            draw_arrow(vis, final_to_direction(final),
                       520, arrow_y, (0, 255, 0), 'cmd')

            text_color = (0, 0, 255) if in_corner else (255, 255, 255)
            cv2.putText(vis,
                f"{'CORNER' if in_corner else 'NORMAL'} | "
                f"mag:{magnitude:+.2f} | px:{pixels}",
                (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.55, text_color, 2)
            cv2.putText(vis,
                f"conf:{conf:.2f} | label:{label} | "
                f"rope:{rope_side} | agree:{consecutive_agreement}/{consecutive_required}",
                (10, 55), cv2.FONT_HERSHEY_SIMPLEX, 0.45,
                (255, 255, 255), 1)
            cv2.putText(vis,
                f"last_turn:{last_turn} | "
                f"corner_frames:{corner_frames:02d} | "
                f"align_count:{align_count}",
                (10, 75), cv2.FONT_HERSHEY_SIMPLEX, 0.45,
                (200, 200, 200), 1)

            if alignment:
                cv2.putText(vis, "ALIGNING",
                            (672 - 120, 30),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.7,
                            (255, 255, 0), 2)

            display_color.value = bgr8_to_jpeg(vis)
            annotated_writer.write(vis)

            elapsed = time.time() - t_start
            time.sleep(max(0, 0.1 - elapsed))

    except KeyboardInterrupt:
        pass
    finally:
        robot.stop()
        annotated_writer.release()
        print(f"Saved: run_{timestamp}.avi")
        print("Stopped")

In [ ]:
import torch
import torch.nn as nn
from torchvision import models, transforms
from pathlib import Path

# --- Config ---
DATA_DIR   = Path("dataset/split")
BATCH_SIZE = 32
EPOCHS     = 20
LR         = 1e-3
DEVICE     = torch.device("mps" if torch.backends.mps.is_available() 
                          else "cuda" if torch.cuda.is_available() 
                          else "cpu")
print(f"Using device: {DEVICE}")

# --- Transforms ---
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3),
    transforms.RandomRotation(10),
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

# --- Datasets ---
'''
train_dataset = datasets.ImageFolder(DATA_DIR / "train", transform=train_transforms)
val_dataset   = datasets.ImageFolder(DATA_DIR / "val",   transform=val_transforms)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)}")
print(f"Classes: {train_dataset.classes}")
'''

# --- Model ---
class LaneClassifier(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()
        backbone = models.resnet18(weights=None)
        backbone.load_state_dict(torch.load(
            #REPO_ROOT / 
            "resnet18-f37072fd.pth",
            map_location="cpu"
        ))
        
        for param in backbone.parameters():
            param.requires_grad = False
            
        in_features = backbone.fc.in_features
        backbone.fc = nn.Sequential(
            nn.Linear(in_features, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )
        self.model = backbone

    def forward(self, x):
        return self.model(x)

In [ ]:
DEVICE     = torch.device("mps" if torch.backends.mps.is_available() 
                          else "cuda" if torch.cuda.is_available() 
                          else "cpu")
model = LaneClassifier(num_classes=3).to(DEVICE)
# Load best weights
model.load_state_dict(torch.load("best_model.pth", map_location=DEVICE))
model.eval()
display(sidebyside)
run_robot(model, robot, DEVICE)

In [ ]:
'''
#IGNORE DEVELOPMENT CODE FOR CODE DEUBGGING AND VISUALISATION PURPOSES ONLY
import cv2
import numpy as np
from collections import deque

def draw_alignment_indicator(vis, magnitude, align_threshold, alignment, w, h,line=None):
    """
    Draws a horizontal bar showing rope offset from centre.
    Centre line = robot is aligned.
    Bar fills left or right to show how far off centre.
    Arrow shows correction direction.
    """
    if not alignment and abs(magnitude) < align_threshold:
        return

    bar_y      = 100
    bar_w      = 200
    bar_h      = 12
    bar_x      = w // 2 - bar_w // 2
    centre_x   = w // 2

    # Background bar
    cv2.rectangle(vis, (bar_x, bar_y),
                  (bar_x + bar_w, bar_y + bar_h),
                  (50, 50, 50), -1)

    # Fill showing offset — magnitude -1 to +1
    fill_pixels = int(magnitude * (bar_w // 2))
    if fill_pixels > 0:
        # Rope to right — fill right of centre
        cv2.rectangle(vis,
                      (centre_x, bar_y),
                      (centre_x + fill_pixels, bar_y + bar_h),
                      (0, 100, 255), -1)
    elif fill_pixels < 0:
        # Rope to left — fill left of centre
        cv2.rectangle(vis,
                      (centre_x + fill_pixels, bar_y),
                      (centre_x, bar_y + bar_h),
                      (255, 100, 0), -1)

    # Centre marker
    cv2.line(vis, (centre_x, bar_y - 4),
             (centre_x, bar_y + bar_h + 4),
             (255, 255, 255), 2)

    # Bar outline
    cv2.rectangle(vis, (bar_x, bar_y),
                  (bar_x + bar_w, bar_y + bar_h),
                  (200, 200, 200), 1)

    # Correction arrow
    arrow_x = centre_x + fill_pixels
    if magnitude < -align_threshold:
        # Need to go left
        cv2.arrowedLine(vis,
                        (centre_x, bar_y + bar_h + 20),
                        (centre_x - 40, bar_y + bar_h + 20),
                        (255, 100, 0), 2, tipLength=0.4)
        cv2.putText(vis, f"ALIGN LEFT  mag:{magnitude:+.2f}",
                    (bar_x, bar_y - 8),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255, 100, 0), 1)
    elif magnitude > align_threshold:
        # Need to go right
        cv2.arrowedLine(vis,
                        (centre_x, bar_y + bar_h + 20),
                        (centre_x + 40, bar_y + bar_h + 20),
                        (0, 100, 255), 2, tipLength=0.4)
        cv2.putText(vis, f"ALIGN RIGHT mag:{magnitude:+.2f}",
                    (bar_x, bar_y - 8),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 100, 255), 1)
    else:
        cv2.putText(vis, f"CENTRED     mag:{magnitude:+.2f}",
                    (bar_x, bar_y - 8),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 255, 0), 1)

    # Draw tilted line showing rope lean direction
    if line is not None:
        vx, vy, x0, y0 = line
        lean_len = 40
        if abs(vy) > 1e-6:
            # Normalise direction vector
            length = np.sqrt(vx**2 + vy**2)
            nx     = vx / length
            ny     = vy / length
            start  = (int(centre_x - nx * lean_len),
                      int(bar_y + bar_h//2 - ny * lean_len))
            end    = (int(centre_x + nx * lean_len),
                      int(bar_y + bar_h//2 + ny * lean_len))
            cv2.line(vis, start, end, (0, 255, 255), 2)

model = LaneClassifier(num_classes=3).to(DEVICE)
model.load_state_dict(torch.load(
    str(REPO_ROOT / "best_model.pth"),
    map_location=DEVICE
))

# ============================================================
# SIMULATE ON VIDEO
# ============================================================
def simulate_on_video(model, avi_path, device,
                      conf_threshold        = 0.85,
                      micro_threshold       = 0.10,
                      micro_frames          = 2,
                      max_spin_frames       = 35,
                      align_threshold       = 0.20,
                      consecutive_required  = 9,
                      output_path           = None):

    avi_path = Path(avi_path)
    model.eval()

    score_history         = deque(maxlen=micro_frames)
    consecutive_agreement = 0
    align_count           = 0
    in_corner             = False
    corner_frames         = 0
    last_turn             = None
    normal_frames         = 0
    last_rope_side = None

    cap    = cv2.VideoCapture(str(avi_path))
    width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    print(f"Simulating: {avi_path.name} ({total} frames)")

    if output_path:
        fourcc = cv2.VideoWriter_fourcc(*'XVID')
        writer = cv2.VideoWriter(str(output_path), fourcc, 10, (width, height))

    def draw_arrow(img, direction, ox, oy, color, label):
        length = 50
        if direction == 'left':
            end = (ox - length, oy)
        elif direction == 'right':
            end = (ox + length, oy)
        else:
            end = (ox, oy - length)
        if direction in ('left', 'right', 'straight'):
            cv2.arrowedLine(img, (ox, oy), end, color, 2, tipLength=0.3)
        cv2.putText(img, label, (ox - 20, oy + 15),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.4, color, 1)

    def final_to_direction(final):
        if final in ('slight_left', 'left', 'exit_corner'):
            return 'left'
        elif final in ('slight_right', 'right'):
            return 'right'
        else:
            return 'straight'

    decisions = []
    frame_idx = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        if len(frame.shape) == 3 and frame.shape[2] == 4:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGRA2BGR)

        h = frame.shape[0]

        # --- ResNet ---
        label, conf, _ = classify_frame(model, frame, device)

        # --- Geometry ---
        cropped   = crop_frame(frame)
        mask      = detect_yellow(cropped)
        pixels    = int(np.sum(mask > 0))
        line      = fit_line(mask)
        magnitude = get_steering_magnitude(line, cropped.shape)
        direction, score, _ = get_line_direction(line, mask, cropped.shape)
        score_history.append(magnitude)
        rope_side = get_rope_side(mask)

        if rope_side is not None:
            last_rope_side = rope_side

        # Fallback chain — rope_side → last_rope_side → last_turn → right
        fallback = rope_side or last_rope_side or last_turn or 'right'
        
        # --- Consecutive agreement counter ---
        if rope_side is not None and \
           label in ('left', 'right') and \
           rope_side == label and \
           conf >= conf_threshold:
            consecutive_agreement += 1
            last_turn = label  # update direction every agreeing frame
        else:
            consecutive_agreement = 0

        # --- Normal frame counter ---
        if not in_corner:
            normal_frames += 1
        else:
            normal_frames = 0

        # --- Corner trigger ---
        corner_trigger = (
            consecutive_agreement >= consecutive_required and
            not in_corner
        )
        # ============================================================
        # STATE
        # ============================================================
        final     = 'straight'
        alignment = False

        if not in_corner and corner_trigger:
            in_corner             = True
            corner_frames         = 0
            align_count           = 0
            consecutive_agreement = 0
            print(f"CORNER → {last_turn} | f:{frame_idx} | "
                  f"conf:{conf:.2f}")

        if in_corner:
            corner_frames += 1
            final = last_turn

            if abs(magnitude) < align_threshold:
                align_count += 1
                if align_count >= 3:
                    in_corner     = False
                    align_count   = 0
                    normal_frames = 0
                    last_turn     = None
                    final         = 'exit_corner'
                    print(f"EXIT corner | f:{frame_idx}")
            else:
                align_count = 0

            if corner_frames > max_spin_frames:
                in_corner     = False
                align_count   = 0
                normal_frames = 0
                # Use current rope_side or last_rope_side as fallback
                final         = f'search_{fallback}'
                print(f"SPIN TIMEOUT | f:{frame_idx} | fallback:{fallback}")

        else:
            # Straight following — line magnitude
            recent = list(score_history)
            if len(recent) == micro_frames and \
               all(s < -micro_threshold for s in recent):
                final     = 'slight_left'
                alignment = True
            elif len(recent) == micro_frames and \
                 all(s > micro_threshold for s in recent):
                final     = 'slight_right'
                alignment = True
            else:
                final = 'straight'

        state = 'CORNER' if in_corner else 'NORMAL'

        decisions.append({
            'frame':                 frame_idx,
            'state':                 state,
            'resnet':                label,
            'conf':                  conf,
            'magnitude':             magnitude,
            'pixels':                pixels,
            'direction':             direction,
            'final':                 final,
            'last_turn':             last_turn,
            'rope_side':             rope_side,
            'consecutive_agreement': consecutive_agreement,
            'alignment':             alignment,
            'corner_frames':         corner_frames,
            'align_count':           align_count,
        })
    
        # --- Draw overlay ---
        vis = frame.copy()
        vis_bottom = vis[h//2:, :]
        vis_bottom[mask > 0] = [0, 255, 255]
        vis[h//2:, :] = vis_bottom

        if line is not None:
            vx, vy, x0, y0 = line
            hc, wc = cropped.shape[:2]
            if abs(vy) > 1e-6:
                x_top    = int(x0 + (0 - y0)  * (vx / vy))
                x_bottom = int(x0 + (hc - y0) * (vx / vy))
                cv2.line(vis, (x_top, h//2),
                         (x_bottom, h), (0, 165, 255), 2)

        arrow_y = h - 40
        line_dir = direction or 'straight'
        draw_arrow(vis, line_dir, 150, arrow_y, (0, 165, 255), 'line')
        draw_arrow(vis, label,    336, arrow_y, (255, 200, 0),
                   f'resnet {conf:.2f}')
        draw_arrow(vis, final_to_direction(final), 520, arrow_y, (0, 255, 0), 'cmd'),

        text_color = (0, 0, 255) if in_corner else (255, 255, 255)

        cv2.putText(vis,
            f"{'CORNER' if in_corner else 'NORMAL'} | "
            f"mag:{magnitude:+.2f} | px:{pixels}",
            (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.55, text_color, 2)
        cv2.putText(vis,
            f"conf:{conf:.2f} | label:{label} | "
            f"rope:{rope_side} | agree:{consecutive_agreement}/{consecutive_required}",
            (10, 55), cv2.FONT_HERSHEY_SIMPLEX, 0.45,
            (255, 255, 255), 1)
        cv2.putText(vis,
            f"last_turn:{last_turn} | "
            f"corner_frames:{corner_frames:02d} | "
            f"align_count:{align_count}",
            (10, 75), cv2.FONT_HERSHEY_SIMPLEX, 0.45,
            (200, 200, 200), 1)

        if alignment:
            cv2.putText(vis, "ALIGNING",
                        (width - 120, 30),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7,
                        (255, 255, 0), 2)

        draw_alignment_indicator(
            vis, magnitude, align_threshold,
            alignment, width, height, line=line
        )

        if output_path:
            writer.write(vis)

        frame_idx += 1

    cap.release()
    if output_path:
        writer.release()
        print(f"Saved to {output_path}")

    return decisions

'''

In [ ]:
#IGNORE DEVELOPMENT CODE FOR CODE DEUBGGING AND VISUALISATION PURPOSES ONLY
'''
corner_starts = []
corner_ends   = []
prev_state    = 'NORMAL'

for d in clockwise_decisions:
    if d['state'] == 'CORNER' and prev_state == 'NORMAL':
        corner_starts.append(d['frame'])
    if d['state'] == 'NORMAL' and prev_state == 'CORNER':
        corner_ends.append(d['frame'])
    prev_state = d['state']

print(f"Number of corners: {len(corner_starts)}")
for i, (s, e) in enumerate(zip(corner_starts, corner_ends)):
    duration = e - s
    corner_d = clockwise_decisions[s]
    print(f"  Corner {i+1}: frame {s}→{e} | "
          f"duration:{duration} | "
          f"turn:{corner_d['last_turn']}")

print(f"\nAvg duration: {sum(e-s for s,e in zip(corner_starts,corner_ends))/max(len(corner_starts),1):.1f} frames")

print(clockwise_df['state'].value_counts())
print(f"Total frames: {len(clockwise_decisions)}")
print(f"\nState distribution:")
print(clockwise_df['state'].value_counts())
print(f"\nAlignment frames: {clockwise_df['alignment'].sum()}")
print(f"\nFinal command distribution:")
print(clockwise_df['final'].value_counts())
print(f"Corner spin frames: {clockwise_df[clockwise_df['state']=='CORNER_SPIN'].shape[0]}")
print(f"Post corner frames: {clockwise_df[clockwise_df['state']=='POST_CORNER'].shape[0]}")

print(anticlockwise_df['state'].value_counts())
print(f"Total frames: {len(anticlockwise_decisions)}")
print(f"\nState distribution:")
print(anticlockwise_df['state'].value_counts())
print(f"\nAlignment frames: {anticlockwise_df['alignment'].sum()}")
print(f"\nFinal command distribution:")
print(anticlockwise_df['final'].value_counts())
print(f"Corner spin frames: {anticlockwise_df[anticlockwise_df['state']=='CORNER_SPIN'].shape[0]}")
print(f"Post corner frames: {anticlockwise_df[anticlockwise_df['state']=='POST_CORNER'].shape[0]}")
'''